## Beispiel an MAGNIFICA HUMANITAS, OF HIS HOLINESS, POPE LEO XIV
Deutsch: https://www.vatican.va/content/leo-xiv/de/encyclicals/documents/20260515-magnifica-humanitas.html

Englisch: https://www.vatican.va/content/leo-xiv/en/encyclicals/documents/20260515-magnifica-humanitas.html

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))

In [2]:
import requests
from bs4 import BeautifulSoup
import os
import re
from pprint import pprint
import json
import pandas as pd
import ast
import string
from collections import Counter
import matplotlib.pyplot as plt

import spacy
from spacy.matcher import DependencyMatcher
from spacy import displacy

nlp = spacy.load('de_core_news_lg') # oder de_core_news_sm

example_path = "example_data"
try:
    os.mkdir(example_path)
    print(f"Directory '{example_path}' created successfully.")
except FileExistsError:
    print(f"Directory '{example_path}' already exists.")

Directory 'example_data' already exists.


In [ ]:
url = "https://www.vatican.va/content/leo-xiv/de/encyclicals/documents/20260515-magnifica-humanitas.html"

response = requests.get(url)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")

# Remove script and style elements
for tag in soup(["script", "style"]):
    tag.decompose()

text = soup.get_text(separator="\n")

# Clean up blank lines
lines = [line.strip() for line in text.splitlines() if line]
start_index = lines.index("EINLEITUNG")
lines = lines[start_index:]
end_index = lines.index("Gegeben zu Rom, bei Sankt Peter, am 15. Mai des Jahres 2026, des zweiten meines Pontifikats.")+1
lines = [re.sub(r'\[\d+\]', '', l)  for l in lines[:end_index]]
lines = [x for x in lines if x.strip()]
text = " ".join(lines).split(".")
print(f"Extracted {len(text)} text segments from the document.")
pprint(text[:5])

output_path = f"{example_path}/magnifica_humanitas.json"
with open(rf"{output_path}", 'w') as f:
    json.dump(text, f)

print(f"Saved to {output_path} ({len(lines)} lines)")


In [3]:
url_en = "https://www.vatican.va/content/leo-xiv/en/encyclicals/documents/20260515-magnifica-humanitas.html"

response = requests.get(url_en)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")

for tag in soup(["script", "style"]):
    tag.decompose()
text_raw_en = soup.get_text(separator="\n")

lines_eng = [line.strip() for line in text_raw_en.splitlines() if line.strip()]
# "INTRODUCTION" appears twice (TOC + body); take the last occurrence
start_index = len(lines_eng) - 1 - lines_eng[::-1].index("INTRODUCTION")
lines_eng = lines_eng[start_index:]
end_index = next(i for i, l in enumerate(lines_eng) if l.startswith("Given in Rome, at Saint Peter")) + 1
lines_en = [re.sub(r'\[\d+\]', '', l) for l in lines_eng[:end_index]]
lines_en = [x for x in lines_en if x.strip()]
text_en = " ".join(lines_en).split(".")
print(f"Extracted {len(text_en)} text segments from the English document.")

output_path_en = f"{example_path}/magnifica_humanitas_en.json"
with open(rf"{output_path_en}", 'w') as f:
    json.dump(text_en, f)

print(f"Saved to {output_path_en} ({len(lines_en)} lines)")

Extracted 1583 text segments from the English document.
Saved to example_data/magnifica_humanitas_en.json (945 lines)


## Filtering 

In [4]:
# preprocessing auxiliary functions

stopwords = open("config_files/spacy_stopwords.txt").read().replace("{", '[').replace("}",']')
stopwords = ast.literal_eval(stopwords)

# this one is not needed if working on Sentence_spacy
def remove_punctuation(corpus):
    cleaned_corpus = corpus.translate(str.maketrans('', '', string.punctuation))
    return(cleaned_corpus)

# remove stop words from text
def remove_stop_words_from_text(cleaned_corpus):
    out = [word for word in cleaned_corpus if not word in stopwords]
    out_text = ' '.join(out)
    return out_text  # , out

# search with stemm/lemma
def get_lemma(sent):
    doc = nlp(sent)
    return [token.lemma for token in doc]
    

# AI-terms
ki_terms = open(f"config_files/ki_terms.txt", "r").read()
ki_terms = [x for x in ki_terms.split("'") if x.strip()]
regex_ki = "".join(["\\b"+x.replace(" ","\\s")+"\\b|" for x in ki_terms])[:-1]
print(regex_ki)

ai_terms = open(f"config_files/ai_terms.txt", "r").read()
ai_terms = [x for x in ai_terms.split("'") if x.strip()]
regex_ai = "".join(["\\b"+x.replace(" ","\\s")+"\\b|" for x in ai_terms])[:-1]
print(regex_ai)

\bpalm\b|\blms\b|\bllama\b|\bTransformer\b|\bSprachmodelle\b|\bSprachmodell\b|\bgpt\b|\bplms\b|\bvortrainierte\sSprachmodelle\b|\bgpt-2\b|\bxlnet\b|\bgroße\sSprachmodelle\b|\bllms\b|\bgpt-3\b|\bStiftungsmodell\b|\bgpt-neo\b|\bgpt-j\b|\bchatgpt\b|\bgpt-4\b|\bKI\b|\bKünstliche\sIntelligenz\b|\bChatGPT\b|\bAI\b|\bCopilot\b|\bGemini\b|\bChatbot\b|\bLLM\b|\bChat-GPT\b|\bSprachmodell\b|\bKI-Modell\b|\bKI-System\b|\bKI-Tool\b|\bArtificial\sIntelligence\b|\bKI-Anwendung\b
\bpalm\b|\blms\b|\bllama\b|\bTransformern\b|\bLanguage\smodels\b|\bLanguage\smodel\b|\bgpt\b|\bplms\b|\bpre-trained\slanguage\smodels\b|\bgpt-2\b|\bxlnet\b|\blarge\slanguage\smodels\b|\bllms\b|\bgpt-3\b|\bFoundation\smodel\b|\bgpt-neo\b|\bgpt-j\b|\bchatgpt\b|\bgpt-4\b|\bAI\b|\bArtificial\sintelligence\b|\bChatGPT\b|\bAI\b|\bCopilot\b|\bGemini\b|\bChatbot\b|\bLLM\b|\bChat\sGPT\b|\bLanguage\smodel\b|\bAI\smodel\b|\bAI\ssystem\b|\bAI\stool\b|\bArtificial\sIntelligence\b|\bAI\sapplication\b


In [ ]:
sentences_ger = [remove_punctuation(line.lower()) for line in text if re.search(regex_ki, line)]
sentences_en = [remove_punctuation(line.lower()) for line in text_en if re.search(regex_ai, line)]
print(f"Found {len(sentences_ger)} sentences in German and {len(sentences_en)} sentences in English containing KI terms.")

#save
with open(f'{example_path}/German_KI_sentences.json', 'w') as f:
    json.dump(sentences_ger, f)
with open(f"{example_path}}/English_KI_sentences.json", "w") as f:
    json.dump(sentences_en, f)

pd.DataFrame({"german": pd.Series(sentences_ger), "english": pd.Series(sentences_en)}).to_csv("example_data/KI_sentences.csv", index=False)

Found 66 sentences in German and 62 sentences in English containing KI terms.


## Data Exploration

In [ ]:
# percentage of ai sentences

# tfidf

In [ ]:
# top words per group

# helper to obtain tokens from an element
def tokens_from_item(item, nlp):
    # spaCy Token lists / Docs / Spans 
    if hasattr(item, "pos_") or (hasattr(item, "__iter__") and not isinstance(item, str) and len(item) and hasattr(item[0], "pos_")):
        # item is a Token, Doc, Span, or list of Tokens
        return list(item)
    
# number and type of verbs using tokens_from_item()
def count_POS(df_col, nlp, pos_tag): # Verb, ADJ, etc.
    tag_count = 0
    tag_types = Counter()
    for text in df_col:
        if len(text)>1:
            tokens = tokens_from_item(text, nlp)
            #print(text,tokens)
            for token in tokens:
                if token.pos_ == pos_tag:
                    tag_count += 1
                    tag_types[token.lemma_] += 1
    print(f"Total {pos_tag}s: {tag_count}")
    print(f"{pos_tag} types:")
    for tag, count in tag_types.most_common():
        print(f"  {tag}: {count}")
    # plot top 20 types
    top_tags = tag_types.most_common(20)
    tags, counts = zip(*top_tags)
    plt.figure(figsize=(10,5))
    plt.bar(tags, counts,color='skyblue')
    plt.title(f'Top {pos_tag}', fontsize=15)
    plt.xlabel(f'{pos_tag} Lemma', fontsize=15)
    #plt.ylabel('Anzahl', fontsize = 18)
    plt.xticks(rotation=90, size=15)
    plt.yticks(size=15)
    plt.tight_layout()
    plt.show()
    plt.savefig(os.path.join(example_path, f'top_{pos_tag}_types.png')) 

In [ ]:
count_POS(sentences_ger, nlp, 'VERB')

In [ ]:
count_POS(sentences_ger, nlp, 'ADJ')

In [ ]:
count_POS(sentences_ger, nlp, 'PROPN')

In [ ]:
count_POS(sentences_ger, nlp, 'NOUN')

## AnthroScore 

In [6]:
# generate input string for AnthroScore for both languages
terminal_input_eng = f'''python3 anthroscore/src/get_anthroscore.py --input_file example_data/KI_sentences.csv--text_column_name english --entities {ai_terms} --output_file example_data/eng_scores.csv --text_id_name id '''
terminal_input_eng

"python3 anthroscore/src/get_anthroscore.py --input_file example_data/KI_sentences.csv--text_column_name english --entities ['palm', 'lms', 'llama', 'Transformern', 'Language models', 'Language model', 'gpt', 'plms', 'pre-trained language models', 'gpt-2', 'xlnet', 'large language models', 'llms', 'gpt-3', 'Foundation model', 'gpt-neo', 'gpt-j', 'chatgpt', 'gpt-4', 'AI', 'Artificial intelligence', 'ChatGPT', 'AI', 'Copilot', 'Gemini', 'Chatbot', 'LLM', 'Chat GPT', 'Language model', 'AI model', 'AI system', 'AI tool', 'Artificial Intelligence', 'AI application'] --output_file example_data/eng_scores.csv --text_id_name id "

In [7]:
# generate input string for AnthroScore for both languages
terminal_input_ger= f'''python3 anthroscore/src/get_anthroscore.py --input_file example_data/KI_sentences.json --text_column_name german --entities {ki_terms} --output_file example_data/ger_scores.csv --text_id_name id '''
terminal_input_ger

"python3 anthroscore/src/get_anthroscore.py --input_file example_data/KI_sentences.json --text_column_name german --entities ['palm', 'lms', 'llama', 'Transformer', 'Sprachmodelle', 'Sprachmodell', 'gpt', 'plms', 'vortrainierte Sprachmodelle', 'gpt-2', 'xlnet', 'große Sprachmodelle', 'llms', 'gpt-3', 'Stiftungsmodell', 'gpt-neo', 'gpt-j', 'chatgpt', 'gpt-4', 'KI', 'Künstliche Intelligenz', 'ChatGPT', 'AI', 'Copilot', 'Gemini', 'Chatbot', 'LLM', 'Chat-GPT', 'Sprachmodell', 'KI-Modell', 'KI-System', 'KI-Tool', 'Artificial Intelligence', 'KI-Anwendung'] --output_file example_data/ger_scores.csv --text_id_name id "

In [8]:
# read results

# plot results

## RegEx and SemEx

In [9]:
# import generated_data/cognitive_v_lexicon.xlsx
cognitive_v_lexicon = pd.read_excel("config_files/cognitive_v_lexicon.xlsx")
#cognitive_v_lexicon.head()

In [10]:
# choose and build lexicon/regex for anthropomorphic verbs
base_verbs = cognitive_v_lexicon['Unnamed: 0'].tolist()
synonyms_all = []
for syn_str in cognitive_v_lexicon['synonyms']:
    synonyms_all.extend(ast.literal_eval(syn_str))

verb_synonyms = [x.lower().strip() for x in base_verbs + synonyms_all]
print(f"found {len(verb_synonyms)} verb synonyms:", verb_synonyms)
anthro_regex = "".join(["\\b"+x.replace(" ","\\s")+"\\b|" for x in verb_synonyms])[:-1]
print(anthro_regex)

# import methods from Anthrop_search.ipynb

# run on sentences from german encyclical

found 75 verb synonyms: ['kennen', 'denken', 'glauben', 'verstehen', 'erinnern', 'vergessen', 'raten', 'vorgeben', 'träumen', 'meinen', 'vermuten', 'behaupten', 'fühlen', 'annehmen', 'auskennen', 'bekannt sein', 'befreundet sein', 'meinen', 'denken', 'vermuten', 'annehmen', 'vertrauen', 'einfallen', 'entsinnen', 'wiedererkennen', 'empfehlen', 'nahelegen', 'vorschlagen', 'zureden', 'aufdecken', 'auflösen', 'enträtseln', 'knacken', 'lösen', 'herausbekommen', 'herauskriegen', 'phantasieren', 'pennen', 'ratzen', 'schlafen', 'schlummern', 'glauben', 'sagen', 'bedeuten', 'bezeichnen', 'annehmen', 'assumieren', 'glauben', 'konjizieren', 'meinen', 'präsumieren', 'supponieren', 'beteuern', 'versichern', 'durchsetzen', 'abtasten', 'berühren', 'betasten', 'erspüren', 'ertasten', 'spüren', 'tasten', 'verspüren', 'empfinden', 'vermuten', 'unterstellen', 'glauben', 'spekulieren', 'denken', 'entgegennehmen', 'in empfang nehmen', 'empfangen', 'akzeptieren', 'sich kümmern', 'sich widmen']
\bkennen\b|\b

In [11]:
# plot results as in Anthrop_search.ipynb (import mehtods to plot)

## Run through models MetaphorBERT and SentenceMetaphorBERT

In [12]:
# import models and methods to run on sentences from german encyclical

In [13]:
# try to run BasicBERT on englisch encyclical (AI and non-AI seperated)

## Use BasicBERT on english encyclical

In [ ]:
# run model on english sentences (ai/ non-ai)

# plot results

In [ ]:
# compare with results from our models
# !! comparability restricted due to differing target languages !!